In [ ]:
from scipy.optimize import linprog
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# HELPER FUNCTIONS
# =============================================================================
def travel_time(distance_m, speed=1.2):
    """Travel time in seconds. Default speed = corridor (1.2 m/s)."""
    return round(distance_m / speed)

def corridor_cap(width_m):
    """Corridor flow capacity: 60 persons/min per metre width."""
    return round(width_m * 600)

def stair_cap(width_m, evac_minutes=15):
    """
    Stair total throughput capacity for a static MCF formulation.
    SFPE: 40 persons/min per metre width.
    Multiply by evacuation duration (default 15 min) to get total persons
    that can pass through the stairwell over the full evacuation.
    """
    return round(width_m * 40 * evac_minutes)

def door_cap(width_m):
    """Door flow capacity: 60 persons/min per metre clear width."""
    return round(width_m * 600)

SPEED_CORRIDOR = 1.2   # m/s -- horizontal walking
SPEED_STAIR    = 0.6   # m/s -- stair descent
SPEED_DOOR     = 0.5   # m/s -- passing through a doorway

# =============================================================================
# 1. NODE DEFINITIONS
# =============================================================================
ROOMS = [
    # Ground Floor
    "GF_FH106", "GF_FH107", "GF_FH108", "GF_FH110",
    "GF_MTLABA", "GF_MTLABB", "GF_GS", "GF_PSR", "GF_FR", "GF_CSR",
    # Second Floor
    "SF_FH201A", "SF_FH201B", "SF_FH202", "SF_FH205", "SF_FH206", "SF_FH207",
    "SF_RE", "SF_CSAR", "SF_AVRA", "SF_AVRB",
    "SF_FH208", "SF_FH209", "SF_FH210", "SF_MMR", "SF_STD",
    "SF_OCL2", "SF_SL", "SF_OCL", "SF_CTA", "SF_CTB", "SF_CTC",
    "SF_DO", "SF_FR",
    # Third Floor
    "TF_FH301A", "TF_FH302", "TF_FH303", "TF_FH304", "TF_FH305",
    "TF_FH306", "TF_FH307", "TF_FH308", "TF_FH309",
    "TF_FH311A", "TF_FH311B", "TF_FHCL", "TF_OAC", "TF_FHTL",
    "TF_FH314", "TF_FH315", "TF_FH316", "TF_FH317",
    "TF_DO", "TF_FR",
]

CORRIDORS = [
    # Ground-floor junctions
    "GF_JUNC_LEFT", "GF_JUNC_CENTER", "GF_JUNC_RIGHT",
    # Second-floor junctions
    "SF_JUNC_LEFT", "SF_JUNC_CENTER", "SF_JUNC_RIGHT",
    # Third-floor junctions
    "TF_JUNC_LEFT", "TF_JUNC_CENTER", "TF_JUNC_RIGHT",
    # Stairwell intermediate nodes
    # _T2S = third-floor side landing; _S2G = second-floor side landing
    "LEFT_STAIR_T2S", "CENTER_STAIR_T2S", "RIGHT_STAIR_T2S",
    "LEFT_STAIR_S2G", "CENTER_STAIR_S2G", "RIGHT_STAIR_S2G",
]

EXITS = [
    "GF_MAINEXIT1", "GF_MAINEXIT2", "GF_MAINEXIT3",
    "GF_MAINEXIT4", "GF_MAINEXIT5", "GF_MAINEXIT6", "GF_MAINEXIT7",
]

EVAC_AREAS   = ["EA_FRONT", "EA_LEFT", "EA_RIGHT"]
SUPER_SOURCE = "s*"
SUPER_SINK   = "t*"
DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday"]

# =============================================================================
# 2. EVACUATION AREA CAPACITIES
# =============================================================================
DELTA = 0.54   # m2 per person (SFPE standard)
EVAC_AREA_DATA = {
    "EA_FRONT": 2800.0,
    "EA_LEFT":   270.0,
    "EA_RIGHT":  185.0,
}
EVAC_CAP = {a: int(EVAC_AREA_DATA[a] / DELTA) for a in EVAC_AREAS}

# =============================================================================
# 3. DAILY OCCUPANCY
# =============================================================================
DAILY_SUPPLY = {
    "Monday": {
        "GF_FH106": 53, "GF_FH107": 56, "GF_FH108": 38, "GF_FH110": 59,
        "GF_MTLABA": 45, "GF_MTLABB": 41, "GF_GS": 12, "GF_PSR": 5,
        "GF_FR": 40, "GF_CSR": 5,
        "SF_FH201A": 47, "SF_FH201B": 60, "SF_FH202": 61, "SF_FH205": 29,
        "SF_FH206": 49, "SF_FH207": 49, "SF_RE": 44, "SF_CSAR": 67,
        "SF_AVRA": 46, "SF_AVRB": 36, "SF_FH208": 47, "SF_FH209": 56,
        "SF_FH210": 34, "SF_MMR": 51, "SF_STD": 0, "SF_OCL2": 53,
        "SF_SL": 47, "SF_OCL": 51, "SF_CTA": 53, "SF_CTB": 56,
        "SF_CTC": 56, "SF_DO": 6, "SF_FR": 30,
        "TF_FH301A": 55, "TF_FH302": 53, "TF_FH303": 50, "TF_FH304": 50,
        "TF_FH305": 51, "TF_FH306": 56, "TF_FH307": 51, "TF_FH308": 55,
        "TF_FH309": 46, "TF_FH311A": 57, "TF_FH311B": 47, "TF_FHCL": 55,
        "TF_OAC": 53, "TF_FHTL": 49, "TF_FH314": 47, "TF_FH315": 42,
        "TF_FH316": 43, "TF_FH317": 53, "TF_DO": 5, "TF_FR": 35,
    },
    "Tuesday": {
        "GF_FH106": 53, "GF_FH107": 49, "GF_FH108": 42, "GF_FH110": 59,
        "GF_MTLABA": 45, "GF_MTLABB": 44, "GF_GS": 12, "GF_PSR": 5,
        "GF_FR": 40, "GF_CSR": 5,
        "SF_FH201A": 59, "SF_FH201B": 61, "SF_FH202": 44, "SF_FH205": 46,
        "SF_FH206": 50, "SF_FH207": 37, "SF_RE": 34, "SF_CSAR": 47,
        "SF_AVRA": 47, "SF_AVRB": 45, "SF_FH208": 48, "SF_FH209": 57,
        "SF_FH210": 34, "SF_MMR": 56, "SF_STD": 51, "SF_OCL2": 51,
        "SF_SL": 48, "SF_OCL": 52, "SF_CTA": 48, "SF_CTB": 49,
        "SF_CTC": 51, "SF_DO": 6, "SF_FR": 30,
        "TF_FH301A": 31, "TF_FH302": 49, "TF_FH303": 53, "TF_FH304": 47,
        "TF_FH305": 40, "TF_FH306": 49, "TF_FH307": 57, "TF_FH308": 46,
        "TF_FH309": 53, "TF_FH311A": 54, "TF_FH311B": 46, "TF_FHCL": 38,
        "TF_OAC": 53, "TF_FHTL": 43, "TF_FH314": 57, "TF_FH315": 49,
        "TF_FH316": 43, "TF_FH317": 53, "TF_DO": 5, "TF_FR": 35,
    },
    "Wednesday": {
        "GF_FH106": 53, "GF_FH107": 33, "GF_FH108": 38, "GF_FH110": 59,
        "GF_MTLABA": 43, "GF_MTLABB": 44, "GF_GS": 12, "GF_PSR": 5,
        "GF_FR": 40, "GF_CSR": 5,
        "SF_FH201A": 59, "SF_FH201B": 48, "SF_FH202": 47, "SF_FH205": 60,
        "SF_FH206": 60, "SF_FH207": 44, "SF_RE": 44, "SF_CSAR": 44,
        "SF_AVRA": 61, "SF_AVRB": 45, "SF_FH208": 51, "SF_FH209": 44,
        "SF_FH210": 29, "SF_MMR": 46, "SF_STD": 50, "SF_OCL2": 49,
        "SF_SL": 49, "SF_OCL": 53, "SF_CTA": 49, "SF_CTB": 47,
        "SF_CTC": 44, "SF_DO": 6, "SF_FR": 30,
        "TF_FH301A": 55, "TF_FH302": 47, "TF_FH303": 49, "TF_FH304": 38,
        "TF_FH305": 55, "TF_FH306": 56, "TF_FH307": 46, "TF_FH308": 43,
        "TF_FH309": 47, "TF_FH311A": 47, "TF_FH311B": 43, "TF_FHCL": 42,
        "TF_OAC": 51, "TF_FHTL": 57, "TF_FH314": 57, "TF_FH315": 47,
        "TF_FH316": 41, "TF_FH317": 50, "TF_DO": 5, "TF_FR": 35,
    },
    "Thursday": {
        "GF_FH106": 56, "GF_FH107": 59, "GF_FH108": 39, "GF_FH110": 53,
        "GF_MTLABA": 45, "GF_MTLABB": 41, "GF_GS": 12, "GF_PSR": 5,
        "GF_FR": 40, "GF_CSR": 5,
        "SF_FH201A": 47, "SF_FH201B": 53, "SF_FH202": 61, "SF_FH205": 56,
        "SF_FH206": 56, "SF_FH207": 56, "SF_RE": 47, "SF_CSAR": 46,
        "SF_AVRA": 47, "SF_AVRB": 59, "SF_FH208": 44, "SF_FH209": 53,
        "SF_FH210": 52, "SF_MMR": 49, "SF_STD": 53, "SF_OCL2": 53,
        "SF_SL": 45, "SF_OCL": 56, "SF_CTA": 36, "SF_CTB": 35,
        "SF_CTC": 56, "SF_DO": 6, "SF_FR": 30,
        "TF_FH301A": 47, "TF_FH302": 45, "TF_FH303": 43, "TF_FH304": 41,
        "TF_FH305": 43, "TF_FH306": 51, "TF_FH307": 46, "TF_FH308": 47,
        "TF_FH309": 49, "TF_FH311A": 51, "TF_FH311B": 49, "TF_FHCL": 40,
        "TF_OAC": 50, "TF_FHTL": 57, "TF_FH314": 47, "TF_FH315": 57,
        "TF_FH316": 41, "TF_FH317": 51, "TF_DO": 5, "TF_FR": 35,
    },
    "Friday": {
        "GF_FH106": 46, "GF_FH107": 34, "GF_FH108": 53, "GF_FH110": 53,
        "GF_MTLABA": 45, "GF_MTLABB": 44, "GF_GS": 12, "GF_PSR": 5,
        "GF_FR": 40, "GF_CSR": 5,
        "SF_FH201A": 53, "SF_FH201B": 59, "SF_FH202": 61, "SF_FH205": 46,
        "SF_FH206": 56, "SF_FH207": 46, "SF_RE": 42, "SF_CSAR": 56,
        "SF_AVRA": 53, "SF_AVRB": 44, "SF_FH208": 51, "SF_FH209": 53,
        "SF_FH210": 0, "SF_MMR": 46, "SF_STD": 53, "SF_OCL2": 53,
        "SF_SL": 49, "SF_OCL": 50, "SF_CTA": 41, "SF_CTB": 48,
        "SF_CTC": 53, "SF_DO": 6, "SF_FR": 30,
        "TF_FH301A": 47, "TF_FH302": 47, "TF_FH303": 39, "TF_FH304": 43,
        "TF_FH305": 54, "TF_FH306": 46, "TF_FH307": 50, "TF_FH308": 47,
        "TF_FH309": 55, "TF_FH311A": 50, "TF_FH311B": 57, "TF_FHCL": 40,
        "TF_OAC": 51, "TF_FHTL": 51, "TF_FH314": 47, "TF_FH315": 47,
        "TF_FH316": 55, "TF_FH317": 48, "TF_DO": 5, "TF_FR": 35,
    },
    "Saturday": {
        "GF_FH106": 53, "GF_FH107": 0, "GF_FH108": 0, "GF_FH110": 53,
        "GF_MTLABA": 30, "GF_MTLABB": 0, "GF_GS": 12, "GF_PSR": 2,
        "GF_FR": 20, "GF_CSR": 2,
        "SF_FH201A": 0, "SF_FH201B": 0, "SF_FH202": 0, "SF_FH205": 45,
        "SF_FH206": 0, "SF_FH207": 45, "SF_RE": 0, "SF_CSAR": 0,
        "SF_AVRA": 0, "SF_AVRB": 0, "SF_FH208": 53, "SF_FH209": 53,
        "SF_FH210": 0, "SF_MMR": 0, "SF_STD": 47, "SF_OCL2": 30,
        "SF_SL": 43, "SF_OCL": 0, "SF_CTA": 53, "SF_CTB": 0,
        "SF_CTC": 0, "SF_DO": 2, "SF_FR": 15,
        "TF_FH301A": 0, "TF_FH302": 53, "TF_FH303": 0, "TF_FH304": 55,
        "TF_FH305": 0, "TF_FH306": 0, "TF_FH307": 0, "TF_FH308": 0,
        "TF_FH309": 0, "TF_FH311A": 57, "TF_FH311B": 49, "TF_FHCL": 36,
        "TF_OAC": 51, "TF_FHTL": 49, "TF_FH314": 47, "TF_FH315": 47,
        "TF_FH316": 43, "TF_FH317": 50, "TF_DO": 2, "TF_FR": 15,
    },
}

# =============================================================================
# 4. EDGE DEFINITIONS
# =============================================================================
def E(u, v, cap, cost):
    return (u, v, cap, cost)

ONE_FLOOR_STAIR = travel_time(3.8, SPEED_STAIR)  # ~6 s per floor

EDGES_BASE = [
    # =========================================================================
    # GROUND FLOOR
    # =========================================================================
    E("GF_FH106",  "GF_JUNC_LEFT",    door_cap(1.2), travel_time(6.0)),
    E("GF_FH107",  "GF_JUNC_LEFT",    door_cap(1.2), travel_time(8.0)),
    E("GF_FH108",  "GF_JUNC_LEFT",    door_cap(1.0), travel_time(10.0)),
    E("GF_MTLABA", "GF_JUNC_LEFT",    door_cap(1.2), travel_time(12.0)),
    E("GF_MTLABB", "GF_JUNC_LEFT",    door_cap(1.2), travel_time(14.0)),
    E("GF_GS",     "GF_JUNC_LEFT",    door_cap(1.0), travel_time(5.0)),
    E("GF_PSR",    "GF_JUNC_LEFT",    door_cap(0.9), travel_time(6.0)),
    E("GF_FR",     "GF_JUNC_LEFT",    door_cap(1.0), travel_time(7.0)),
    E("GF_CSR",    "GF_JUNC_LEFT",    door_cap(0.9), travel_time(6.0)),
    E("GF_FH110",  "GF_JUNC_CENTER",  door_cap(1.2), travel_time(6.0)),
    E("GF_JUNC_LEFT",   "GF_JUNC_CENTER", corridor_cap(2.4), travel_time(20.0)),
    E("GF_JUNC_CENTER", "GF_JUNC_LEFT",   corridor_cap(2.4), travel_time(20.0)),
    E("GF_JUNC_CENTER", "GF_JUNC_RIGHT",  corridor_cap(2.4), travel_time(20.0)),
    E("GF_JUNC_RIGHT",  "GF_JUNC_CENTER", corridor_cap(2.4), travel_time(20.0)),
    E("GF_JUNC_LEFT",   "GF_MAINEXIT1",   door_cap(1.8), travel_time(3.0)),
    E("GF_JUNC_LEFT",   "GF_MAINEXIT2",   door_cap(1.8), travel_time(4.0)),
    E("GF_JUNC_LEFT",   "GF_MAINEXIT3",   door_cap(1.8), travel_time(8.0)),
    E("GF_JUNC_CENTER", "GF_MAINEXIT4",   door_cap(2.0), travel_time(3.0)),
    E("GF_JUNC_CENTER", "GF_MAINEXIT5",   door_cap(2.0), travel_time(3.0)),
    E("GF_JUNC_RIGHT",  "GF_MAINEXIT6",   door_cap(1.8), travel_time(3.0)),
    E("GF_JUNC_RIGHT",  "GF_MAINEXIT7",   door_cap(1.8), travel_time(5.0)),
    E("GF_MAINEXIT1", "EA_FRONT", door_cap(1.8), travel_time(15.0)),
    E("GF_MAINEXIT2", "EA_FRONT", door_cap(1.8), travel_time(12.0)),
    E("GF_MAINEXIT3", "EA_LEFT",  door_cap(1.8), travel_time(10.0)),
    E("GF_MAINEXIT4", "EA_LEFT",  door_cap(2.0), travel_time(8.0)),
    E("GF_MAINEXIT5", "EA_LEFT",  door_cap(2.0), travel_time(10.0)),
    E("GF_MAINEXIT6", "EA_RIGHT", door_cap(1.8), travel_time(10.0)),
    E("GF_MAINEXIT7", "EA_RIGHT", door_cap(1.8), travel_time(12.0)),
    # =========================================================================
    # SECOND FLOOR
    # =========================================================================
    E("SF_FH201A", "SF_JUNC_LEFT",   door_cap(1.2), travel_time(6.0)),
    E("SF_FH201B", "SF_JUNC_LEFT",   door_cap(1.2), travel_time(8.0)),
    E("SF_FH202",  "SF_JUNC_LEFT",   door_cap(1.2), travel_time(10.0)),
    E("SF_FH205",  "SF_JUNC_LEFT",   door_cap(1.2), travel_time(14.0)),
    E("SF_FH206",  "SF_JUNC_LEFT",   door_cap(1.2), travel_time(16.0)),
    E("SF_FH207",  "SF_JUNC_LEFT",   door_cap(1.2), travel_time(18.0)),
    E("SF_RE",     "SF_JUNC_LEFT",   door_cap(1.0), travel_time(5.0)),
    E("SF_CSAR",   "SF_JUNC_LEFT",   door_cap(1.5), travel_time(7.0)),
    E("SF_AVRA",   "SF_JUNC_CENTER", door_cap(1.5), travel_time(6.0)),
    E("SF_AVRB",   "SF_JUNC_CENTER", door_cap(1.5), travel_time(6.0)),
    E("SF_FH208",  "SF_JUNC_CENTER", door_cap(1.2), travel_time(8.0)),
    E("SF_FH209",  "SF_JUNC_CENTER", door_cap(1.2), travel_time(10.0)),
    E("SF_FH210",  "SF_JUNC_CENTER", door_cap(1.2), travel_time(10.0)),
    E("SF_MMR",    "SF_JUNC_CENTER", door_cap(1.2), travel_time(6.0)),
    E("SF_STD",    "SF_JUNC_CENTER", door_cap(1.2), travel_time(6.0)),
    E("SF_OCL2",   "SF_JUNC_RIGHT",  door_cap(1.2), travel_time(8.0)),
    E("SF_SL",     "SF_JUNC_RIGHT",  door_cap(1.2), travel_time(10.0)),
    E("SF_OCL",    "SF_JUNC_RIGHT",  door_cap(1.2), travel_time(12.0)),
    E("SF_CTA",    "SF_JUNC_RIGHT",  door_cap(1.2), travel_time(14.0)),
    E("SF_CTB",    "SF_JUNC_RIGHT",  door_cap(1.2), travel_time(16.0)),
    E("SF_CTC",    "SF_JUNC_RIGHT",  door_cap(1.2), travel_time(18.0)),
    E("SF_DO",     "SF_JUNC_RIGHT",  door_cap(0.9), travel_time(4.0)),
    E("SF_FR",     "SF_JUNC_RIGHT",  door_cap(1.0), travel_time(5.0)),
    E("SF_JUNC_LEFT",   "SF_JUNC_CENTER", corridor_cap(2.4), travel_time(22.0)),
    E("SF_JUNC_CENTER", "SF_JUNC_LEFT",   corridor_cap(2.4), travel_time(22.0)),
    E("SF_JUNC_CENTER", "SF_JUNC_RIGHT",  corridor_cap(2.4), travel_time(22.0)),
    E("SF_JUNC_RIGHT",  "SF_JUNC_CENTER", corridor_cap(2.4), travel_time(22.0)),
    E("SF_JUNC_LEFT",   "LEFT_STAIR_S2G",   stair_cap(1.5), travel_time(5.0)),
    E("SF_JUNC_CENTER", "CENTER_STAIR_S2G", stair_cap(1.5), travel_time(5.0)),
    E("SF_JUNC_RIGHT",  "RIGHT_STAIR_S2G",  stair_cap(1.5), travel_time(5.0)),
    E("LEFT_STAIR_S2G",   "GF_JUNC_LEFT",   stair_cap(1.5), ONE_FLOOR_STAIR),
    E("CENTER_STAIR_S2G", "GF_JUNC_CENTER", stair_cap(1.5), ONE_FLOOR_STAIR),
    E("RIGHT_STAIR_S2G",  "GF_JUNC_RIGHT",  stair_cap(1.5), ONE_FLOOR_STAIR),
    # =========================================================================
    # THIRD FLOOR
    # =========================================================================
    E("TF_FH301A", "TF_JUNC_LEFT",   door_cap(1.2), travel_time(6.0)),
    E("TF_FH302",  "TF_JUNC_LEFT",   door_cap(1.2), travel_time(8.0)),
    E("TF_FH303",  "TF_JUNC_LEFT",   door_cap(1.2), travel_time(10.0)),
    E("TF_FH304",  "TF_JUNC_LEFT",   door_cap(1.2), travel_time(12.0)),
    E("TF_FH305",  "TF_JUNC_LEFT",   door_cap(1.2), travel_time(14.0)),
    E("TF_FH306",  "TF_JUNC_LEFT",   door_cap(1.2), travel_time(16.0)),
    E("TF_FH307",  "TF_JUNC_LEFT",   door_cap(1.2), travel_time(18.0)),
    E("TF_FH308",  "TF_JUNC_CENTER", door_cap(1.2), travel_time(6.0)),
    E("TF_FH309",  "TF_JUNC_CENTER", door_cap(1.2), travel_time(8.0)),
    E("TF_FH311A", "TF_JUNC_CENTER", door_cap(1.2), travel_time(10.0)),
    E("TF_FH311B", "TF_JUNC_CENTER", door_cap(1.2), travel_time(10.0)),
    E("TF_FHCL",   "TF_JUNC_CENTER", door_cap(1.5), travel_time(6.0)),
    E("TF_OAC",    "TF_JUNC_CENTER", door_cap(1.2), travel_time(6.0)),
    E("TF_FHTL",   "TF_JUNC_CENTER", door_cap(1.2), travel_time(8.0)),
    E("TF_FH314",  "TF_JUNC_RIGHT",  door_cap(1.2), travel_time(8.0)),
    E("TF_FH315",  "TF_JUNC_RIGHT",  door_cap(1.2), travel_time(10.0)),
    E("TF_FH316",  "TF_JUNC_RIGHT",  door_cap(1.2), travel_time(12.0)),
    E("TF_FH317",  "TF_JUNC_RIGHT",  door_cap(1.2), travel_time(14.0)),
    E("TF_DO",     "TF_JUNC_RIGHT",  door_cap(0.9), travel_time(4.0)),
    E("TF_FR",     "TF_JUNC_RIGHT",  door_cap(1.0), travel_time(5.0)),
    E("TF_JUNC_LEFT",   "TF_JUNC_CENTER", corridor_cap(2.4), travel_time(22.0)),
    E("TF_JUNC_CENTER", "TF_JUNC_LEFT",   corridor_cap(2.4), travel_time(22.0)),
    E("TF_JUNC_CENTER", "TF_JUNC_RIGHT",  corridor_cap(2.4), travel_time(22.0)),
    E("TF_JUNC_RIGHT",  "TF_JUNC_CENTER", corridor_cap(2.4), travel_time(22.0)),
    E("TF_JUNC_LEFT",   "LEFT_STAIR_T2S",   stair_cap(1.5), travel_time(5.0)),
    E("TF_JUNC_CENTER", "CENTER_STAIR_T2S", stair_cap(1.5), travel_time(5.0)),
    E("TF_JUNC_RIGHT",  "RIGHT_STAIR_T2S",  stair_cap(1.5), travel_time(5.0)),
    E("LEFT_STAIR_T2S",   "SF_JUNC_LEFT",   stair_cap(1.5), ONE_FLOOR_STAIR),
    E("CENTER_STAIR_T2S", "SF_JUNC_CENTER", stair_cap(1.5), ONE_FLOOR_STAIR),
    E("RIGHT_STAIR_T2S",  "SF_JUNC_RIGHT",  stair_cap(1.5), ONE_FLOOR_STAIR),
]

# =============================================================================
# 5. SOLVER
# =============================================================================
def build_and_solve(day_name, verbose=True):
    supply_data = DAILY_SUPPLY[day_name]
    supply = {room: supply_data.get(room, 0) for room in ROOMS}
    S = sum(supply.values())

    if verbose:
        print("=" * 65)
        print(f" DAY: {day_name} | Total Evacuees (S) = {S}")
        print("=" * 65)

    total_cap = sum(EVAC_CAP.values())
    if total_cap < S:
        print(f" [INFEASIBLE] Evac capacity ({total_cap}) < Supply ({S})\n")
        return None

    ss_edges = []
    for room in ROOMS:
        if supply[room] > 0:
            ss_edges.append((SUPER_SOURCE, room, supply[room], 0))
    for area in EVAC_AREAS:
        ss_edges.append((area, SUPER_SINK, EVAC_CAP[area], 0))

    ALL_EDGES = EDGES_BASE + ss_edges
    n_edges   = len(ALL_EDGES)
    ALL_NODES = ROOMS + CORRIDORS + EXITS + EVAC_AREAS + [SUPER_SOURCE, SUPER_SINK]
    node_idx  = {n: i for i, n in enumerate(ALL_NODES)}
    n_nodes   = len(ALL_NODES)

    c_obj  = np.array([cost for (_, _, _, cost) in ALL_EDGES], dtype=float)
    bounds = [(0, cap) for (_, _, cap, _) in ALL_EDGES]

    A_eq = np.zeros((n_nodes, n_edges))
    b_eq = np.zeros(n_nodes)

    for e_idx, (u, v, cap, cost) in enumerate(ALL_EDGES):
        if u in node_idx: A_eq[node_idx[u], e_idx] -= 1
        if v in node_idx: A_eq[node_idx[v], e_idx] += 1

    b_eq[node_idx[SUPER_SOURCE]] = -S
    b_eq[node_idx[SUPER_SINK]]   =  S

    result = linprog(c_obj, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')

    if result.status != 0:
        print(f" [SOLVER] {result.message}\n")
        return None

    flow_vals = result.x
    obj_val   = result.fun

    out_edges = {n: [] for n in ALL_NODES}
    in_edges  = {n: [] for n in ALL_NODES}
    for i, (u, v, cap, cost) in enumerate(ALL_EDGES):
        if u in out_edges: out_edges[u].append(i)
        if v in in_edges:  in_edges[v].append(i)

    if verbose:
        print(f"\n Optimal Total Cost Z = {obj_val:.2f} (person-seconds)\n")
        print(f" {'Edge':<50} {'Cap':>5} {'Flow':>7} {'Cost':>6} {'Util%':>7}")
        print(" " + "-" * 80)

    bottlenecks = []
    for i, (u, v, cap, cost) in enumerate(ALL_EDGES):
        if u == SUPER_SOURCE or v == SUPER_SINK:
            continue
        fv = flow_vals[i]
        if fv > 0.01:
            util = (fv / cap * 100) if cap > 0 else 0
            tag  = " BOTTLENECK" if util >= 90 else ""
            if util >= 90:
                bottlenecks.append((u, v, cap, fv, util))
            edge_str = f"{u} -> {v}"
            print(f" {edge_str:<50} {cap:>5} {fv:>7.1f} {cost:>6} {util:>6.1f}%{tag}")

    print(f"\n {'Evacuation Area':<20} {'Capacity':>10} {'Received':>10} {'Util%':>8}")
    print(" " + "-" * 52)
    for area in EVAC_AREAS:
        received = sum(flow_vals[i] for i in in_edges[area])
        util = received / EVAC_CAP[area] * 100
        print(f" {area:<20} {EVAC_CAP[area]:>10} {received:>10.1f} {util:>7.1f}%")

    if bottlenecks:
        print(f"\n BOTTLENECK EDGES (>=90% utilization):")
        for u, v, cap, fv, util in bottlenecks:
            print(f"   {u} -> {v} | Flow: {fv:.1f}/{cap} ({util:.1f}%)")
    else:
        print(f"\n No bottlenecks detected.")
    print()

    return {
        'day': day_name, 'total_supply': S, 'optimal_cost': obj_val,
        'flow_vals': flow_vals, 'edges': ALL_EDGES,
        'in_edges': in_edges, 'out_edges': out_edges,
    }

# =============================================================================
# 6. OPTIMAL ROUTE TRACER
# =============================================================================
def get_optimal_route(result, start_room, verbose=True):
    if result is None:
        return None
    edges, flow_vals = result['edges'], result['flow_vals']
    adj = {}
    for i, (u, v, cap, cost) in enumerate(edges):
        adj.setdefault(u, []).append((v, i))
    path, visited, current = [start_room], {start_room}, start_room
    terminals = set(EVAC_AREAS + [SUPER_SINK])
    while current not in terminals:
        candidates = [(v, idx) for v, idx in adj.get(current, []) if v not in visited]
        if not candidates:
            break
        best_v, best_idx = max(candidates, key=lambda x: flow_vals[x[1]])
        if flow_vals[best_idx] < 0.01:
            break
        path.append(best_v)
        visited.add(best_v)
        current = best_v
    display_path = [n for n in path if n != SUPER_SINK]
    if verbose:
        print(f" {start_room:<15} -> {' -> '.join(display_path[1:])}")
    return display_path

# =============================================================================
# 7. MAIN -- RUN ALL DAYS
# =============================================================================
print("=" * 65)
print(" FEDERIZO HALL -- EVACUATION FLOW OPTIMIZER")
print(" Capacitated Multi-Source Multi-Sink Minimum Cost Flow")
print(" BulSU Main Campus")
print("=" * 65)

print(f"\n Evacuation Area Capacities:")
for a, cap in EVAC_CAP.items():
    print(f"  {a:<12} {EVAC_AREA_DATA[a]:>7} m2 -> {cap:>5} persons")
print(f"  {'TOTAL':<12} {'':>7} {sum(EVAC_CAP.values()):>5} persons\n")

results = {}
for day in DAYS:
    results[day] = build_and_solve(day, verbose=True)

# =============================================================================
# 8. WEEKLY SUMMARY
# =============================================================================
print("\n" + "=" * 58)
print(" WEEKLY SUMMARY")
print(f" {'Day':<12} {'Evacuees':>10} {'Optimal Z':>14} {'Status':>12}")
print(" " + "-" * 52)
for day in DAYS:
    r = results[day]
    if r:
        print(f" {day:<12} {r['total_supply']:>10} {r['optimal_cost']:>14.2f}     FEASIBLE")
    else:
        print(f" {day:<12} {'N/A':>10} {'N/A':>14}   INFEASIBLE")

# =============================================================================
# 9. SAMPLE ROUTE TRACES
# =============================================================================
print("\n" + "=" * 65)
day_to_check = "Monday"
rooms_to_trace = [
    "TF_FH301A", "TF_FH311A",
    "SF_FH201A", "SF_CSAR",
    "GF_FH106",  "GF_MTLABA",
]
print(f" OPTIMAL ROUTES -- {day_to_check}")
print("=" * 65)
r = results.get(day_to_check)
if r:
    for room in rooms_to_trace:
        get_optimal_route(r, room)